In [0]:
#CREATE THE MONITORING RESULTS TABLE
CATALOG = "airbnb_obs"

spark.sql(f""" 
CREATE TABLE IF NOT EXISTS {CATALOG}.monitoring.dq_results (
              run_id                     STRING,        -- ties all rules from one execution together
              check_ts                   TIMESTAMP,     -- when the rule ran
              layer                      STRING,        -- bronze / silver/ gold
              table_name                 STRING,        -- table thw rule checked
              column_name                STRING,        -- column the rule checked (or '*' for table-level)
              rule_name                  STRING,        -- human name of the rule
              rule_type                  STRING,        -- not_null / unique / range / freshness / row_count ...
              failed_records             BIGINT,        -- number of rows that violated the rule
              total_records              BIGINT,        -- rows evaluated
              status                     STRING,        -- PASS / FAIL
              details                    STRING        -- free-text / JSON context
          ) USING DELTA
          """) 
print("monitoring.dq_results ready")

In [0]:
# REUSABLE CHECK FUNCTIONS, EACH RETURNS A COUNT OF FAILING ROWS
from pyspark.sql import functions as F
from datetime import datetime, timezone
import uuid

def _count(df):
  return df.count()

def check_not_null(df, column):
    return df.filter(F.col(column).isNull() | (F.trim(F.col(column)) == "")).count()
                  
def check_unique(df, column): 
    dupes = df.groupBy(column).count().filter("count > 1")
    return dupes.agg(F.coalesce(F.sum(F.col("count")-1), F.lit(0))).collect()[0][0]
  
def check_range (df, column, min_val=None, max_val=None):
    c = F.col(column).cast("double")
    cond = F.lit(False)
    if min_val is not None: cond = cond | (c < min_val)
    if max_val is not None: cond = cond | (c > max_val)
    return df.filter(cond).count()
  
def check_accepted_values(df, column, allowed):
    return df.filter(~F.col(column).isin(allowed)).count()
  
def check_row_count_min(df, minimum): # table-level: "failed"=1 if the table has too few rows , else 0
    return 1 if df.count() < minimum else 0

In [0]:
# THE RUNNER THAT EXECUTES THE RULES AND WRITES RESULTS

def run_rules(layer, table_name, rules):
    df = spark.table(f"{CATALOG}.{layer}.{table_name}")
    total = df.count()
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6]
    results = []

    for r in rules:
        rtype = r["rule_type"]
        col   = r.get("column", "*")
        if   rtype == "not_null":        failed = check_not_null(df, col)
        elif rtype == "unique":          failed = check_unique(df, col)
        elif rtype == "range":           failed = check_range(df, col, r.get("min"), r.get("max"))
        elif rtype == "accepted_values": failed = check_accepted_values(df, col, r["allowed"])
        elif rtype == "row_count":       failed = check_row_count_min(df, r["min"])
        else: raise ValueError(f"unknown rule_type {rtype}")

        results.append((
            run_id, datetime.now(timezone.utc), layer, table_name, col,
            r["rule_name"], rtype, int(failed), int(total),
            "PASS" if failed == 0 else "FAIL", r.get("details", "")
        ))

    cols = ["run_id","check_ts","layer","table_name","column_name","rule_name",
            "rule_type","failed_records","total_records","status","details"]
    spark.createDataFrame(results, cols) \
         .write.mode("append").saveAsTable(f"{CATALOG}.monitoring.dq_results")

    print(f"✔ {table_name}: ran {len(rules)} rules (run_id={run_id})")
    return run_id

In [0]:
# RULES DECLARATION FOR BRONZE


host_rules = [
    {"rule_name": "host_id not null",   "rule_type": "not_null", "column": "host_id"},
    {"rule_name": "host_id unique",     "rule_type": "unique",   "column": "host_id"},
    {"rule_name": "response_rate 0-100","rule_type": "range",    "column": "response_rate", "min": 0, "max": 100},
    {"rule_name": "hosts row count",    "rule_type": "row_count","min": 100},
]

booking_rules = [
    {"rule_name": "booking_id unique",  "rule_type": "unique",   "column": "booking_id"},
    {"rule_name": "listing_id not null","rule_type": "not_null", "column": "listing_id"},
    {"rule_name": "status valid",       "rule_type": "accepted_values", "column": "booking_status",
     "allowed": ["confirmed", "cancelled"]},
    {"rule_name": "nights 1-30",        "rule_type": "range",    "column": "nights_booked", "min": 1, "max": 30},
]

listing_rules = [
    {"rule_name": "listing_id not null",  "rule_type": "not_null", "column": "listing_id"},
    {"rule_name": "listing_id unique",    "rule_type": "unique",   "column": "listing_id"},
    {"rule_name": "host_id not null",     "rule_type": "not_null", "column": "host_id"},
    {"rule_name": "price > 0",            "rule_type": "range",    "column": "price_per_night", "min": 0.01},
    {"rule_name": "accommodates 1-16",    "rule_type": "range",    "column": "accommodates", "min": 1, "max": 16},
    {"rule_name": "room_type valid",      "rule_type": "accepted_values", "column": "room_type",
     "allowed": ["Entire home", "Private room", "Shared room", "Hotel room"]},
    {"rule_name": "listings row count",   "rule_type": "row_count", "column": "*", "min": 100},
]

run_rules("bronze", "listings", listing_rules)
run_rules("bronze", "hosts", host_rules)
run_rules("bronze", "bookings", booking_rules)

In [0]:
# DATA QUALITY RULES RESULTS

display(
    spark.table(f"{CATALOG}.monitoring.dq_results")
         .orderBy(F.desc("check_ts"))
)